# Diários Oficiais do Rio — Downloader

> Notebook standalone para baixar PDFs dos diários oficiais do Município do Rio de Janeiro de forma automática.

## Por quê?

Quem acompanha política municipal carioca precisa cruzar dois diários oficiais quase todo dia:

- **DOP** — *Diário Oficial da Prefeitura* (`doweb.rio.rj.gov.br`) → atos do Executivo: decretos, gastos, contratos, nomeações.
- **DCM** — *Diário da Câmara Municipal* (`dcmdigital.camara.rj.gov.br`) → atos do Legislativo: projetos de lei, pareceres, discursos, votações.

Os dois portais oferecem download manual no navegador, mas não têm API pública. Esse notebook automatiza a etapa — três modos de uso, uma fonte ou as duas, configurável em uma célula.

## Modos suportados

| Modo | DCM | DOP |
|---|---|---|
| `hoje` | última edição publicada | última edição publicada |
| `edicao` | nº + ano | nº ou data |
| `range` | intervalo de números no mesmo ano | intervalo de números ou de datas |

## Como funciona internamente

- **DOP** → 100% HTTP. A home expõe a edição corrente numa variável JS `DADOS_ULTIMA_DATA`; basta um regex e um GET no endpoint de download.
- **DCM** → Selenium headless. A página renderiza por JavaScript e o formulário de "buscar edição" é interativo (`<select>` de ano + `<input>` de número + botão OK + modal com cadernos). Tem retry automático porque o CDN do DCM ocasionalmente devolve `SSL: UNEXPECTED_EOF_WHILE_READING`.

## Como usar

1. Configure as variáveis na primeira célula de código (`MODO`, `FONTE`, `DESTINO`).
2. Se for a primeira vez, descomente as linhas da célula de **Instalação**.
3. **Run All**. Os PDFs aparecem em `DESTINO`.

---
*Validado em CI (GitHub Actions, Ubuntu + Chrome stable). Parte do projeto [clipping-project](https://github.com/OttoBoop/clipping-project) — pipeline de clipping político carioca.*


## 1. Configuração

A única célula que você precisa editar. Lê de cima pra baixo:

- **MODO** — o que baixar: a edição de hoje, uma específica, ou um intervalo.
- **FONTE** — de qual diário: só DCM, só DOP, ou os dois.
- **EDICAO_\***, **RANGE_\*** — parâmetros usados conforme o modo.
- **DESTINO** — pasta onde os PDFs caem.

### Exemplos rápidos

```python
# Tudo de hoje (mais comum)
MODO, FONTE = "hoje", "ambos"

# DCM edição 108 de 2025
MODO, FONTE = "edicao", "dcm"
EDICAO_NUMERO, EDICAO_ANO = 108, 2025

# DOP do dia 31/03/2025 (precisa estar disponível no portal, veja limitação na Seção 3)
MODO, FONTE = "edicao", "dop"
EDICAO_DATA = "2025-03-31"

# DCM edições 156 a 167 de 2025
MODO, FONTE = "range", "dcm"
RANGE_TIPO, RANGE_INICIO, RANGE_FIM, EDICAO_ANO = "edicao", 156, 167, 2025

# DOP de uma semana
MODO, FONTE = "range", "dop"
RANGE_TIPO, RANGE_INICIO, RANGE_FIM = "data", "2025-03-24", "2025-03-30"
```


In [ ]:
# ==========================================================
#  O QUE BAIXAR
# ==========================================================
MODO   = "hoje"      # "hoje" | "edicao" | "range"
FONTE  = "ambos"     # "dcm"  | "dop"    | "ambos"

# --- usado se MODO == "edicao" ---
EDICAO_NUMERO = 108        # número da edição (obrigatório pro DCM em edicao/range)
EDICAO_ANO    = 2025
EDICAO_DATA   = None       # alternativa pro DOP: "YYYY-MM-DD" ou None

# --- usado se MODO == "range" ---
RANGE_TIPO   = "edicao"    # "edicao" (DCM e DOP) | "data" (só DOP)
RANGE_INICIO = 156         # número de edição (int) OU data "YYYY-MM-DD" (str)
RANGE_FIM    = 158

# ==========================================================
#  ONDE SALVAR
# ==========================================================
DESTINO = "./downloads"    # ex.: "/content/drive/MyDrive/DOs" no Colab


## 2. Instalação

**Quando descomentar:**

| Ambiente | `pip install` | `apt install chromium` |
|---|---|---|
| Google Colab (primeira vez) | sim | sim, mas use `chromium-chromedriver` ou rode em ambiente com Chrome real |
| Máquina local com Chrome instalado | só os pacotes que faltarem | não precisa |
| GitHub Actions (CI) | sim | use a action `browser-actions/setup-chrome@v1` em vez do apt |

**Importante:** o Selenium 4.6+ usa `selenium-manager` internamente, que baixa o ChromeDriver compatível sozinho. Você só precisa garantir que o **binário do Chrome** esteja instalado e acessível.


In [ ]:
# !pip install --quiet requests beautifulsoup4 PyPDF2 selenium pandas
# !apt-get install -y chromium-chromedriver   # necessário apenas pro DCM


### Imports e setup do destino

Tudo é stdlib + 3 libs externas (`requests`, `PyPDF2`, `selenium` — `pandas` só na última célula pra mostrar resumo). O `urllib3.disable_warnings` silencia avisos do CDN do DCM que ocasionalmente cai em fallback `verify=False`.

In [ ]:
import os, re, json, time, shutil, datetime
from urllib.parse import urljoin
import requests
from PyPDF2 import PdfReader, PdfMerger
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.makedirs(DESTINO, exist_ok=True)
print(f"Destino: {os.path.abspath(DESTINO)}")


## 3. DOP — Diário Oficial da Prefeitura

Fonte: <https://doweb.rio.rj.gov.br>

### Estratégia

100% HTTP, sem Selenium. A home do portal entrega o HTML com uma variável JavaScript embutida:

```js
let DADOS_ULTIMA_DATA = {
  "itens": [
    { "id": 14807, "suplemento": "", ... },    // ← caderno principal
    { "id": 14808, "suplemento": "I", ... },   // ← suplementos do mesmo dia
    ...
  ]
};
```

Pegamos via regex, escolhemos o item com `suplemento == ""` (o caderno principal), e baixamos via `GET /portal/edicoes/download/{id}`.

### Funções públicas

| Função | Modo |
|---|---|
| `baixar_dop_hoje()` | `MODO="hoje"` |
| `baixar_dop_por_data("YYYY-MM-DD")` | `MODO="edicao"` com `EDICAO_DATA` |
| `baixar_dop_por_edicao(numero)` | `MODO="edicao"` com `EDICAO_NUMERO` |
| `baixar_dop_range(inicio, fim, tipo)` | `MODO="range"` |

### Limitação conhecida — arquivo histórico

`baixar_dop_por_data` e `baixar_dop_por_edicao` só funcionam para itens que ainda estejam expostos em `DADOS_ULTIMA_DATA` da home. **Datas/edições antigas não são acessíveis por essa rota.** O portal não documenta endpoint público de arquivo histórico.

Pra contornar: abra o portal no navegador, navegue até a edição desejada, inspecione o link `/portal/edicoes/download/{id}` no devtools e use o ID direto chamando `_dop_baixar_pdf(session, id, nome, destino)`.


In [ ]:
DOP_BASE = "https://doweb.rio.rj.gov.br"
DOP_HEADERS = {
    # UA real é necessário — o portal devolve 403 pra UAs vazios/curl
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

def _dop_pegar_itens_home(session):
    """Lê DADOS_ULTIMA_DATA do HTML da home e retorna a lista de itens (cadernos)."""
    r = session.get(DOP_BASE, headers=DOP_HEADERS, timeout=30)
    r.raise_for_status()
    m = re.search(r"let DADOS_ULTIMA_DATA = (\{.*?\});", r.text, re.DOTALL)
    if not m:
        raise RuntimeError("Variável DADOS_ULTIMA_DATA não encontrada na home do DOP.")
    return json.loads(m.group(1)).get("itens", [])

def _dop_baixar_pdf(session, item_id, nome_arquivo, destino):
    """Baixa um PDF pelo ID interno do portal. Valida Content-Type pra evitar
    salvar HTML de erro como .pdf."""
    url = f"{DOP_BASE}/portal/edicoes/download/{item_id}"
    print(f"[DOP] baixando id={item_id} <- {url}")
    r = session.get(url, headers=DOP_HEADERS, timeout=120)
    r.raise_for_status()
    if "application/pdf" not in r.headers.get("Content-Type", "").lower():
        raise RuntimeError(f"Resposta não é PDF (Content-Type={r.headers.get('Content-Type')})")
    caminho = os.path.join(destino, nome_arquivo)
    with open(caminho, "wb") as f:
        f.write(r.content)
    print(f"[DOP] OK {caminho} ({len(r.content)/1024/1024:.2f} MB)")
    return caminho

def baixar_dop_hoje(destino=None):
    """Baixa a edição mais recente publicada do DOP (caderno principal, sem suplemento)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        # caderno principal = suplemento vazio (suplementos têm "I", "II", etc)
        item = next((i for i in itens if i.get("suplemento") == ""), None)
        if not item:
            raise RuntimeError("Nenhuma edição com suplemento vazio encontrada na home.")
        data_str = datetime.date.today().strftime("%Y_%m_%d")
        return _dop_baixar_pdf(s, item["id"], f"DOP_{data_str}.pdf", destino)

def baixar_dop_por_data(data_iso: str, destino=None):
    """Tenta baixar o DOP de uma data específica (YYYY-MM-DD).

    Procura a data em DADOS_ULTIMA_DATA verificando campos comuns (data,
    data_publicacao, data_edicao). Se não achar, levanta erro com instrução
    pro usuário inspecionar o portal.
    """
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("data", "data_publicacao", "data_edicao"):
                if str(item.get(key, "")).startswith(data_iso) and item.get("suplemento") == "":
                    nome = f"DOP_{data_iso.replace('-', '_')}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP {data_iso} não está exposto em DADOS_ULTIMA_DATA. "
            "Pra datas mais antigas é preciso descobrir o endpoint de arquivo do portal. "
            f"Sugestão: abra {DOP_BASE} no navegador, procure a edição da data desejada "
            "e anote o link de download."
        )

def baixar_dop_por_edicao(numero: int, destino=None):
    """Tenta baixar o DOP pelo número da edição (procura na home)."""
    destino = destino or DESTINO
    with requests.Session() as s:
        itens = _dop_pegar_itens_home(s)
        for item in itens:
            for key in ("numero", "edicao", "numero_edicao"):
                if str(item.get(key, "")) == str(numero) and item.get("suplemento") == "":
                    nome = f"DOP_edicao_{numero}.pdf"
                    return _dop_baixar_pdf(s, item["id"], nome, destino)
        raise RuntimeError(
            f"DOP edição {numero} não está exposta em DADOS_ULTIMA_DATA. "
            "Veja a mensagem de baixar_dop_por_data."
        )

def baixar_dop_range(inicio, fim, tipo: str, destino=None):
    """Baixa um intervalo de DOPs.

    tipo='data':   inicio e fim no formato 'YYYY-MM-DD' (inclusivos).
    tipo='edicao': inicio e fim como inteiros (inclusivos).

    Falhas individuais são logadas mas não interrompem o range.
    """
    destino = destino or DESTINO
    saidas = []
    if tipo == "data":
        d_ini = datetime.date.fromisoformat(str(inicio))
        d_fim = datetime.date.fromisoformat(str(fim))
        d = d_ini
        while d <= d_fim:
            try:
                saidas.append(baixar_dop_por_data(d.isoformat(), destino))
            except Exception as e:
                print(f"[DOP] aviso ({d}): {e}")
            d += datetime.timedelta(days=1)
    elif tipo == "edicao":
        for n in range(int(inicio), int(fim) + 1):
            try:
                saidas.append(baixar_dop_por_edicao(n, destino))
            except Exception as e:
                print(f"[DOP] aviso (edicao {n}): {e}")
    else:
        raise ValueError("tipo deve ser 'data' ou 'edicao'")
    return saidas


## 4. DCM — Diário da Câmara Municipal

Fonte: <https://dcmdigital.camara.rj.gov.br>

### Por que Selenium e não requests?

A home do DCM é uma SPA que renderiza por JavaScript. Um `curl` direto pega só o shell HTML sem os dados. Pra extrair a última edição (regex em `<span id="edAtual">`) ou interagir com o formulário de busca por número, **precisa** de um browser real esperando o JS executar.

### Fluxo por edição específica

1. Abre a home, espera o `<select id="yDcm2">` ficar visível.
2. Seleciona o ano via `Select.select_by_value`.
3. Preenche `<input id="inputEdicao">` com o número da edição.
4. Clica `<button id="btnEd">` (OK).
5. Espera o `<div id="corpoModal">` aparecer com a lista de cadernos.
6. Cada `<figure>` no modal tem um `<a href="/download/{id}">` — extrai todos.
7. Baixa cada caderno via `requests` (com cookies da sessão Selenium se necessário).
8. Se houver múltiplos cadernos, une com `PyPDF2.PdfMerger`.

### Retry — por que existe

O CDN do DCM ocasionalmente devolve **`SSL: UNEXPECTED_EOF_WHILE_READING`** no meio do download dos cadernos. É flakiness do servidor, não do código. A constante `DCM_TENTATIVAS=3` com `DCM_BACKOFF_S=6` segundos resolve sem intervenção manual.

### Funções públicas

| Função | Modo |
|---|---|
| `baixar_dcm_hoje()` | `MODO="hoje"` |
| `baixar_dcm_edicao(ano, numero)` | `MODO="edicao"` |
| `baixar_dcm_range(ano, inicio, fim)` | `MODO="range"` |


In [ ]:
DCM_BASE = "https://dcmdigital.camara.rj.gov.br/"
DCM_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
          "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")
DCM_REQ_HEADERS = {"User-Agent": DCM_UA, "Referer": DCM_BASE,
                   "Accept": "application/pdf,*/*"}

# Retry config: o CDN do DCM as vezes devolve SSL EOF no meio do download.
DCM_TENTATIVAS = 3
DCM_BACKOFF_S = 6

def _dcm_novo_driver():
    """Cria um WebDriver Chrome novo em modo headless.
    O Selenium 4.6+ resolve o chromedriver automaticamente via selenium-manager."""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options as ChromeOptions
    opts = ChromeOptions()
    for arg in ("--headless", "--no-sandbox", "--disable-dev-shm-usage",
                "--disable-gpu", "--window-size=1920,1080"):
        opts.add_argument(arg)
    opts.add_argument(f"--user-agent={DCM_UA}")
    return webdriver.Chrome(options=opts)

def _dcm_retry(fn, *args, label="", **kwargs):
    """Executa fn com até DCM_TENTATIVAS tentativas, com backoff fixo.
    Selenium + DCM são intermitentemente flaky (race entre drivers, SSL EOF do CDN)."""
    ultimo_erro = None
    for tentativa in range(1, DCM_TENTATIVAS + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            ultimo_erro = e
            print(f"[DCM] {label} tentativa {tentativa}/{DCM_TENTATIVAS} falhou: "
                  f"{type(e).__name__}: {str(e)[:120]}")
            if tentativa < DCM_TENTATIVAS:
                time.sleep(DCM_BACKOFF_S)
    raise ultimo_erro

def _dcm_baixar_urls(urls, prefixo, destino):
    """Baixa cada URL como caderno parcial e une em um único PDF final.

    Se vier 1 caderno: renomeia direto pro nome final.
    Se vierem N: unifica com PdfMerger e remove os temporários."""
    parciais = []
    with requests.Session() as s:
        s.headers.update(DCM_REQ_HEADERS)
        for i, url in enumerate(urls, 1):
            try:
                r = s.get(url, timeout=120, verify=False)
                r.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"[DCM] erro baixando {url}: {e}")
                continue
            if "application/pdf" not in r.headers.get("Content-Type", "").lower():
                print(f"[DCM] aviso: {url} não retornou PDF, pulando.")
                continue
            tmp = os.path.join(destino, f"_tmp_{prefixo}_{i}.pdf")
            with open(tmp, "wb") as f:
                f.write(r.content)
            parciais.append(tmp)
    if not parciais:
        raise RuntimeError("Nenhum caderno baixado com sucesso.")
    if len(parciais) == 1:
        final = os.path.join(destino, f"{prefixo}.pdf")
        shutil.move(parciais[0], final)
    else:
        final = os.path.join(destino, f"{prefixo}_unificado.pdf")
        merger = PdfMerger()
        try:
            for p in parciais:
                merger.append(p)
            merger.write(final)
        finally:
            merger.close()
        for p in parciais:
            try: os.remove(p)
            except OSError: pass
    print(f"[DCM] OK {final}")
    return final

def _baixar_dcm_hoje_uma_vez(destino):
    """Implementação sem retry — usada via _dcm_retry."""
    driver = _dcm_novo_driver()
    try:
        driver.get(DCM_BASE)
        time.sleep(5)               # espera o JS terminar de popular a home
        html = driver.page_source
    finally:
        driver.quit()

    # extrai "Última Edição: NN - DD/MM/AAAA" do span renderizado
    m = re.search(
        r'<span id="edAtual">\s*Última Edição:\s*(\d+)\s*</span>\s*<span>\s*-\s*(\d{2}/\d{2}/\d{4})',
        html
    )
    if not m:
        raise RuntimeError("Não consegui extrair número/data da edição mais recente.")
    edicao = m.group(1)
    data_str = m.group(2).replace("/", "_")
    print(f"[DCM] última edição: {edicao} ({data_str})")

    # dois layouts possíveis na home: caderno único (comentário HTML marcador)
    # ou múltiplos cadernos (botões com attr arg="...")
    m_unico = re.search(
        r'<!-- Só tem 01 Caderno -->.*?<a href="(https://dcmdigital\.camara\.rj\.gov\.br/download/[^"]+)"',
        html, re.DOTALL
    )
    if m_unico:
        urls = [m_unico.group(1).strip()]
    else:
        ids = re.findall(
            r'class="btn btn-sm btn-primary btn-busca buscaDcmDoDia".+?arg="(\w+)"',
            html, re.DOTALL
        )
        if not ids:
            raise RuntimeError("Não encontrei links de caderno na home do DCM.")
        urls = [urljoin(DCM_BASE, f"download/{a}") for a in ids]

    return _dcm_baixar_urls(urls, f"DCM_{data_str}_ed{edicao}", destino)

def baixar_dcm_hoje(destino=None):
    """Baixa a última edição publicada do DCM (todos os cadernos do dia, unificados)."""
    destino = destino or DESTINO
    return _dcm_retry(_baixar_dcm_hoje_uma_vez, destino, label="hoje")

def _baixar_dcm_edicao_uma_vez(ano, numero, destino):
    """Implementação sem retry — usada via _dcm_retry."""
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import Select, WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException

    driver = _dcm_novo_driver()
    urls = []
    try:
        driver.get(DCM_BASE)
        wait = WebDriverWait(driver, 40)
        # 1) ano
        wait.until(EC.visibility_of_element_located((By.ID, "yDcm2")))
        time.sleep(2)
        Select(driver.find_element(By.ID, "yDcm2")).select_by_value(str(ano))
        time.sleep(1)
        # 2) numero
        inp = wait.until(EC.visibility_of_element_located((By.ID, "inputEdicao")))
        inp.clear()
        inp.send_keys(str(numero))
        # 3) clica OK e espera o modal
        wait.until(EC.element_to_be_clickable((By.ID, "btnEd"))).click()
        wait.until(EC.visibility_of_element_located((By.ID, "corpoModal")))
        time.sleep(2)
        # 4) coleta links de download (cada figure = 1 caderno)
        try:
            figs = wait.until(EC.presence_of_all_elements_located(
                (By.XPATH, "//div[@id='corpoModal']/figure")))
        except TimeoutException:
            figs = []
        for fig in figs:
            try:
                link = fig.find_element(By.XPATH, ".//a[starts-with(@href, '/download/')]")
                href = link.get_attribute("href")
                if href:
                    urls.append(urljoin(DCM_BASE, href))
            except Exception:
                continue
    finally:
        driver.quit()

    if not urls:
        raise RuntimeError(f"DCM edição {numero}/{ano}: nenhum caderno encontrado.")
    return _dcm_baixar_urls(urls, f"DCM_ed{numero}_ano{ano}", destino)

def baixar_dcm_edicao(ano: int, numero: int, destino=None):
    """Baixa o DCM de uma edição específica (ano + número). Faz retry automático
    em flakiness do Selenium ou SSL EOF do CDN."""
    destino = destino or DESTINO
    return _dcm_retry(_baixar_dcm_edicao_uma_vez, ano, numero, destino,
                      label=f"ed {numero}/{ano}")

def baixar_dcm_range(ano: int, inicio: int, fim: int, destino=None):
    """Baixa um intervalo de edições do DCM (todas no mesmo ano).
    Falhas individuais são logadas mas não interrompem o range."""
    destino = destino or DESTINO
    saidas = []
    for n in range(int(inicio), int(fim) + 1):
        try:
            saidas.append(baixar_dcm_edicao(ano, n, destino))
        except Exception as e:
            print(f"[DCM] aviso (edicao {n}): {e}")
    return saidas


## 5. Runner

Esta célula é o dispatcher: lê as variáveis da Seção 1 e chama as funções certas. Você raramente precisa mexer aqui.

Falha de uma fonte (DOP ou DCM) não derruba a outra — cada bloco está em seu próprio `try/except`. Útil quando, por exemplo, o DCM está com instabilidade mas você quer pelo menos o DOP do dia.


In [ ]:
def rodar():
    saidas = []

    # ---------- DOP ----------
    if FONTE in ("dop", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dop_hoje(DESTINO))
            elif MODO == "edicao":
                # DOP aceita data OU número — data tem precedência se ambas
                # estiverem setadas
                if EDICAO_DATA:
                    saidas.append(baixar_dop_por_data(EDICAO_DATA, DESTINO))
                else:
                    saidas.append(baixar_dop_por_edicao(EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                saidas += baixar_dop_range(RANGE_INICIO, RANGE_FIM, RANGE_TIPO, DESTINO)
            else:
                print(f"[DOP] MODO desconhecido: {MODO}")
        except Exception as e:
            print(f"[DOP] falhou: {e}")

    # ---------- DCM ----------
    if FONTE in ("dcm", "ambos"):
        try:
            if MODO == "hoje":
                saidas.append(baixar_dcm_hoje(DESTINO))
            elif MODO == "edicao":
                saidas.append(baixar_dcm_edicao(EDICAO_ANO, EDICAO_NUMERO, DESTINO))
            elif MODO == "range":
                # DCM só suporta range por número de edição (não tem URL por data)
                if RANGE_TIPO != "edicao":
                    print("[DCM] RANGE_TIPO='data' não se aplica ao DCM, pulando.")
                else:
                    saidas += baixar_dcm_range(EDICAO_ANO, RANGE_INICIO, RANGE_FIM, DESTINO)
            else:
                print(f"[DCM] MODO desconhecido: {MODO}")
        except Exception as e:
            print(f"[DCM] falhou: {e}")

    # filtra Nones caso alguma função tenha retornado vazio em falha tolerante
    return [s for s in saidas if s]

pdfs_baixados = rodar()
print(f"\n{len(pdfs_baixados)} PDF(s) baixado(s).")


## 6. Resultado

Tabela com os arquivos baixados nesta execução. Útil pra confirmar que tudo está coerente antes de subir pra um drive, mandar pra alguém ou alimentar uma próxima etapa de pipeline (extração, resumo, classificação).


In [ ]:
import pandas as pd

linhas = []
for p in pdfs_baixados:
    try:
        n_paginas = len(PdfReader(p).pages)
    except Exception:
        n_paginas = "?"
    linhas.append({
        "arquivo":   os.path.basename(p),
        "tamanho_MB": round(os.path.getsize(p) / 1024 / 1024, 2),
        "paginas":   n_paginas,
        "modificado": datetime.datetime.fromtimestamp(
            os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M"),
    })

if linhas:
    df = pd.DataFrame(linhas)
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))
else:
    print("Nenhum PDF foi baixado nesta execução.")


## 7. Troubleshooting

### `SessionNotCreatedException: no chrome binary at /usr/bin/google-chrome`
Selenium não encontrou o Chrome. Soluções:
- **Colab / Linux:** instale Google Chrome (`apt install google-chrome-stable`) ou aponte explicitamente: `opts.binary_location = "/usr/bin/chromium"`.
- **macOS:** o Chrome em `/Applications/Google Chrome.app` costuma ser detectado automaticamente. Se não, `opts.binary_location = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"`.
- **GitHub Actions:** use `browser-actions/setup-chrome@v1` em vez do apt — o pacote `chromium-browser` do Ubuntu é um stub de snap e não funciona em CI.

### `Variável DADOS_ULTIMA_DATA não encontrada na home do DOP`
O portal pode ter mudado a estrutura. Abra `https://doweb.rio.rj.gov.br` no devtools, procure `DADOS_ULTIMA_DATA` ou variável equivalente no `<script>` da home, ajuste o regex em `_dop_pegar_itens_home`.

### `DOP <data> não está exposto em DADOS_ULTIMA_DATA`
Limitação conhecida — o portal só expõe edições recentes. Pra arquivo histórico, pegue o ID manualmente no devtools do navegador e chame `_dop_baixar_pdf` direto.

### `SSL: UNEXPECTED_EOF_WHILE_READING` no DCM
Flakiness do CDN. O retry interno (`DCM_TENTATIVAS=3`) já trata. Se persistir, aumente: `DCM_TENTATIVAS = 5; DCM_BACKOFF_S = 10`.

### `TimeoutException` no DCM ao buscar `yDcm2`/`inputEdicao`/`corpoModal`
A página não terminou de renderizar a tempo. Aumente o `WebDriverWait(driver, 40)` ou o `time.sleep(5)` na home.

### O PDF salvou mas está vazio / corrompido
O `Content-Type` é validado nas duas pontas, mas se mesmo assim acontecer, abra o arquivo e veja se começa com `%PDF-`. Em geral indica que o servidor devolveu HTML de erro com `Content-Type: application/pdf` (caso raro).

---

## 8. Como adaptar pra outros diários oficiais

O padrão é genérico:

1. **Achar o endpoint de download.** Use o devtools do navegador na aba *Network* enquanto baixa manualmente uma edição.
2. **Descobrir como o portal lista edições.** Variável JS na home? API JSON? Formulário HTML?
3. **Escolher entre HTTP puro ou Selenium.** Se a listagem está no HTML renderizado pelo servidor, `requests` resolve. Se for SPA, Selenium.
4. **Padronizar o nome do arquivo.** Prefira `{ORIGEM}_{data}_{edicao}.pdf` pra ordenar fácil.
5. **Tratar PDFs múltiplos.** Diários costumam ter "cadernos" — junte com `PyPDF2.PdfMerger` ou salve separados, conforme sua necessidade downstream.
6. **Adicionar retry.** Sites de governo são notoriamente instáveis em horário de pico.
